
# RappiPlus | End-to-End Business Analytics Capstone

### Python · SQL · Power BI · Business Intelligence

**Project:** TripleTen Data Analytics Bootcamp — Final Capstone  
**Focus:** Commercial performance, profitability, conversion, and customer retention

## Project Overview

This capstone brings together multiple analytical techniques to evaluate business performance and support data-driven decision-making.

The project covers data preparation, SQL-based analysis, business KPIs, conversion funnels, cohort retention, statistical testing, and business intelligence reporting.

## Analytical Objectives

1. Prepare and validate data for analysis.
2. Evaluate commercial performance and profitability.
3. Analyse customer conversion across the funnel.
4. Investigate customer retention through cohort analysis.
5. Apply statistical methods to assess business outcomes.
6. Translate findings into actionable insights and BI reporting.

## Tools & Methods

- **Python:** Data cleaning, exploration, and statistical analysis
- **SQL:** Data extraction and querying
- **Power BI:** Business intelligence and visual storytelling
- **Analytical methods:** KPI analysis, funnel analysis, cohort retention, and hypothesis testing

---

## Analysis & Methodology



## Data Sources & Analytical Workflow

The RappiPlus analysis integrates multiple business datasets:

- **Orders:** Order transactions, pricing, discounts, and revenue.
- **Product Catalogue:** Product costs, categories, and suppliers.
- **Marketing Spend:** Marketing investment by channel and country.
- **SQL Database:** Events, users, and user activity.
- **Checkout Experiment:** A/B testing results for the checkout interface.

### Analytical Workflow

The project follows six stages:

1. **Data Quality:** Assess data completeness, consistency, and reliability.
2. **Profitability:** Analyse revenue, costs, and profit.
3. **Conversion Funnel:** Identify where users drop off during their journey.
4. **Cohort Retention:** Evaluate whether customers return over time.
5. **Experimentation:** Assess the statistical significance of checkout changes.
6. **Business Intelligence:** Communicate findings through Power BI dashboards.

The objective is to connect operational performance, customer behaviour, and commercial outcomes to support evidence-based business decisions.


---


## 01. Data Loading & Quality Assessment

### 1.1 Data Import & Initial Exploration

**Objective:** Understand the structure, completeness, and reliability of the business datasets before conducting further analysis.

### Data Sources

Three datasets are imported into Python:

- `rappiplus_orders_raw.csv` — Transactional orders data
- `rappiplus_catalog.csv` — Product catalogue and cost information
- `rappiplus_marketing_spend.csv` — Marketing investment data

The datasets are loaded into Pandas DataFrames (`orders`, `catalog`, and `marketing`) for initial exploration.

### Analytical Approach

The initial assessment focuses on understanding dataset structure, reviewing available variables, and identifying potential data-quality issues that could affect subsequent profitability and commercial performance analysis.


In [ ]:
# importar librerías
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
# Load datasets from local CSV files
orders = pd.read_csv("rappiplus_orders_raw.csv")
catalog = pd.read_csv("rappiplus_catalog.csv")
marketing = pd.read_csv("rappiplus_marketing_spend.csv")

In [ ]:
print("Orders:", orders.shape)
print("Catalog:", catalog.shape)
print("Marketing:", marketing.shape)

In [ ]:
# explore datasets
print("\n--- VIEW OF ORDERS --")
print(orders.head())
print(orders.info())

print("\n--- VIEW OF CATALOG --")
print(catalog.head())
print(catalog.info())

print("\n--- VIEW OF MARKETING ---")
print(marketing.head())
print(marketing.info())

---


### 1.2 Data Quality & Validation

**Objective:** Identify and address potential data-quality issues that could affect revenue, cost, and profitability analysis.

The validation process covers three business datasets: orders, product catalogue, and marketing expenditure.

#### Validation Framework

1. **Date Standardisation:** Convert date columns into appropriate datetime formats.
2. **Numerical Validation:** Examine numeric variables for invalid negative or zero values.
3. **Financial Consistency:** Verify the consistency of monetary amounts used in commercial calculations.
4. **Duplicate Detection:** Identify and remove duplicate records where appropriate.
5. **Categorical Consistency:** Review categorical variables for inconsistencies that could affect aggregation and reporting.

These checks establish a more reliable foundation for subsequent profitability analysis and business intelligence reporting.


In [ ]:
# 1. Convert date columns to datetime
if 'fecha_hora_pedido' in orders.columns:
    orders['fecha_hora_pedido'] = pd.to_datetime(
        orders['fecha_hora_pedido'], errors='coerce'
    )


if 'fecha' in marketing.columns:
    marketing['fecha'] = pd.to_datetime(marketing['fecha'])

# 2. Eliminate duplicated rows in all the datasets
orders = orders.drop_duplicates()
catalog = catalog.drop_duplicates()
marketing = marketing.drop_duplicates()

# 3. Validate numeric variables
if 'cantidad' in orders.columns:
    orders = orders[orders['cantidad'] > 0]

if 'precio_unitario' in orders.columns:
    orders = orders[orders['precio_unitario'] >= 0]
    
if 'costo_unitario' in catalog.columns:
    catalog = catalog[catalog['costo_unitario'] >= 0]

# Verify datasets after cleaning
print("Final Dimensions of orders:", orders.shape)
print("Final Dimensions of catalog:", catalog.shape)
print("Final Dimensions of  marketing:", marketing.shape)


In [ ]:
print(orders['fecha_hora_pedido'].dtype)
print(orders['fecha_hora_pedido'].isna().sum())

In [ ]:
orders.to_csv("orders_clean.csv", index=False)
print("Updated orders_clean.csv successfully")

In [ ]:
# export datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

print("¡Files successfully exported into the project's folder! 🚀")

In [ ]:
print("Missing product names:", orders["nombre_producto"].isna().sum())
print("Missing product categories:", orders["categoria_producto"].isna().sum())

In [ ]:
missing_products = orders[orders["nombre_producto"].isna()]

print("Affected orders:", len(missing_products))
print("Net revenue:", round(missing_products["monto_total"].sum(), 2))
print(
    "Gross revenue:",
    round(
        (
            missing_products["cantidad"]
            * missing_products["precio_unitario"]
        ).sum(),
        2
    )
)

In [ ]:
print("ORDERS — Missing Values")
print(orders.isnull().sum())

print("\nCATALOG — Missing Values")
print(catalog.isnull().sum())

print("\nMARKETING — Missing Values")
print(marketing.isnull().sum())

In [ ]:
missing_products = orders[orders["nombre_producto"].isna()]

print("Orders with missing product names:", len(missing_products))
print("Total recorded sales value:", missing_products["monto_total"].sum())

display(missing_products.head())


print(f"Revenue associated with missing product names: ${missing_products['monto_total'].sum():,.2f}")


total_revenue = orders["monto_total"].sum()

missing_revenue_pct = (
    missing_products["monto_total"].sum() / total_revenue
) * 100

print(f"Total recorded revenue: ${total_revenue:,.2f}")
print(f"Revenue with unidentified products: {missing_revenue_pct:.4f}%")

In [ ]:
# Audit orders with missing product categories
missing_categories = orders[
    orders["categoria_producto"].isna()
]

print("Orders with missing categories:", len(missing_categories))
print(
    "Revenue from missing categories: "
    f"${missing_categories['monto_total'].sum():,.2f}"
)
print(
    "Gross revenue from missing categories: "
    f"${(missing_categories['cantidad'] * missing_categories['precio_unitario']).sum():,.2f}"
)

---


## 02. Commercial Performance & Profitability Analysis

### 2.1 Business KPI Assessment

**Objective:** Evaluate RappiPlus's financial performance, profitability, and sales behaviour through key commercial indicators.

The analysis integrates transactional orders, product catalogue data, and marketing expenditure to assess business performance.

### Part A — Revenue & Profitability

The financial assessment addresses four key questions:

1. What is the total revenue generated?
2. What are the total product costs?
3. How much has been invested in marketing?
4. Is the business profitable after accounting for product costs and marketing expenditure?

**Key metrics:** Total Revenue, Total Costs, Marketing Spend, and Profit.

### Part B — Sales & Product Performance

The commercial assessment examines:

1. Average order value.
2. Average number of products per order.
3. Best-selling products by quantity.
4. Marketing expenditure by acquisition channel.

These indicators help identify revenue drivers, purchasing patterns, product demand, and opportunities for commercial optimisation.

### Analytical Methodology

Order-level transactions are combined with product catalogue information to connect sales revenue with product costs.

The resulting dataset supports profitability calculations and subsequent reporting in Power BI.


In [ ]:
# Audit product cost coverage before profitability calculations
cost_audit = orders.merge(
    catalog,
    on="nombre_producto",
    how="left",
    indicator=True
)

print("Orders without a catalogue match:")
print((cost_audit["_merge"] == "left_only").sum())

print("\nOrders with missing product costs:")
print(cost_audit["costo_unitario"].isna().sum())

# Calculate product costs only where catalogue data is available
cost_audit["calculated_cost"] = (
    cost_audit["cantidad"] * cost_audit["costo_unitario"]
)

known_product_cost = cost_audit["calculated_cost"].sum()

print(f"Known product costs: ${known_product_cost:,.2f}")
print(
    "Orders with unknown product costs:",
    cost_audit["calculated_cost"].isna().sum()
)


In [ ]:
gross_revenue = (
    orders["cantidad"] * orders["precio_unitario"]
).sum()

recorded_revenue = orders["monto_total"].sum()

print(f"Revenue before discounts: ${gross_revenue:,.2f}")
print(f"Recorded revenue: ${recorded_revenue:,.2f}")
print(f"Difference: ${gross_revenue - recorded_revenue:,.2f}")


total_discounts = orders["monto_descuento"].sum()

print(f"Total recorded discounts: ${total_discounts:,.2f}")
print(
    f"Unexplained difference: "
    f"${gross_revenue - recorded_revenue - total_discounts:,.2f}"
)

In [ ]:

# BUSINESS PROFITABILITY & SALES BEHAVIOUR

# 1. Merge orders with the product catalog to retrieve unit costs
orders_with_costs = pd.merge(
    orders, catalog, on="nombre_producto", how="left"
)

# 2. Calculate revenue and known product costs for each transaction
orders_with_costs["total_revenue"] = (
    orders_with_costs["cantidad"] * orders_with_costs["precio_unitario"]
)

orders_with_costs["total_cost"] = (
    orders_with_costs["cantidad"] * orders_with_costs["costo_unitario"]
)

# 3. Calculate estimated profitability
# Net sales revenue after recorded discounts
revenue_total = orders["monto_total"].sum()

# Missing product costs are excluded from the sum
costo_productos_total = orders_with_costs["total_cost"].sum()
gasto_marketing_total = marketing["gasto"].sum()

# This is an estimated profit upper bound, not accounting net income.
# Product costs are unavailable for 30 orders.
# Other operating expenses are not included.
profit_total = revenue_total - (
    costo_productos_total + gasto_marketing_total
)

print("--- BUSINESS PROFITABILITY ---")
print(f"Net Sales Revenue: ${revenue_total:,.2f}")
print(f"Known Product Costs: ${costo_productos_total:,.2f}")
print(f"Marketing Expenditure: ${gasto_marketing_total:,.2f}")
print(f"Estimated Profit After Marketing: ${profit_total:,.2f}")
print(
    "Note: Profit is provisional because some product costs "
    "and other operating expenses are unavailable."
)

# 4. Analyse sales behaviour

# Average order value = net sales revenue / unique orders
ticket_promedio = revenue_total / orders["id_pedido"].nunique()

# Average number of units per order
productos_por_orden = (
    orders.groupby("id_pedido")["cantidad"].sum().mean()
)

# Best-selling product by units sold
producto_mas_vendido = (
    orders.groupby("nombre_producto")["cantidad"].sum().idxmax()
)

cantidad_mas_vendida = (
    orders.groupby("nombre_producto")["cantidad"].sum().max()
)

print("\n--- SALES BEHAVIOUR ---")
print(f"Average Order Value: ${ticket_promedio:,.2f}")
print(f"Average Units per Order: {productos_por_orden:.2f}")
print(
    f"Best-Selling Product: {producto_mas_vendido} "
    f"({cantidad_mas_vendida:,.0f} units)"
)

In [ ]:

print(f"Section 02 revenue: ${revenue_total:,.2f}")
print(f"Section 02 product costs: ${costo_productos_total:,.2f}")
print(f"Section 02 marketing: ${gasto_marketing_total:,.2f}")
print(f"Section 02 estimated profit: ${profit_total:,.2f}")


In [ ]:
# Validate estimated profit using net sales revenue
net_revenue = orders["monto_total"].sum()
known_costs = cost_audit["calculated_cost"].sum()
marketing_costs = marketing["gasto"].sum()

estimated_profit = net_revenue - known_costs - marketing_costs

print(f"Net sales revenue: ${net_revenue:,.2f}")
print(f"Known product costs: ${known_costs:,.2f}")
print(f"Marketing expenditure: ${marketing_costs:,.2f}")
print(f"Estimated profit: ${estimated_profit:,.2f}")


In [ ]:

# Compare profitability before and after marketing costs
gross_profit_estimate = (
    revenue_total - costo_productos_total
)

profit_after_marketing = (
    gross_profit_estimate - gasto_marketing_total
)

print(f"Gross profit estimate: ${gross_profit_estimate:,.2f}")
print(f"Marketing expenditure: ${gasto_marketing_total:,.2f}")
print(f"Profit after marketing: ${profit_after_marketing:,.2f}")


In [ ]:
gross_profit_estimate = revenue_total - costo_productos_total

print(f"Gross profit estimate: ${gross_profit_estimate:,.2f}")
print(f"Marketing expenditure: ${gasto_marketing_total:,.2f}")
print(f"Profit after marketing: ${profit_total:,.2f}")


### Data Quality Note — Unclassified Transactions

The dataset contains **30 orders with missing product names and categories**, representing **$12,262.62 in net sales revenue** and **$12,387.59 in gross revenue before discounts**.

These transactions are retained in the revenue calculations to avoid understating sales. However, their associated product costs cannot be verified.

Consequently, profitability estimates that subtract only recorded product costs may overstate actual profit. The Power BI dashboard separately reports **Verified Gross Profit** for transactions with known product costs.

This distinction ensures transparency between reported sales, known costs, and estimated profitability.

---


## 03. Customer Conversion Funnel Analysis

### 3.1 User Journey & Funnel Construction

**Objective:** Analyse customer interactions across the purchasing journey to identify potential bottlenecks and opportunities to improve the checkout experience.

### Data Source & SQL Methodology

User interaction data is retrieved from the `events` table using SQL queries executed through a PostgreSQL database connection.

The analysis counts distinct users associated with each event type (`nombre_evento`) and presents these events in their expected purchasing-journey order.

### Part A — Funnel Construction

The analysis addresses:

1. How many unique users are recorded at each purchasing-journey stage?
2. How do event-level user counts differ across stages?
3. Which stages show the largest differences in recorded user volume?

These counts provide an overview of customer activity across the purchasing journey.

### Part B — Event-Level User Count Analysis

The analysis examines:

1. Unique-user counts for each event type.
2. Differences in user counts between adjacent stages.
3. The relationship between initial-visit and purchase event counts.

These comparisons help identify potential areas of friction but do not establish verified sequential conversion or individual user drop-off.

### Business Application

The results support exploratory commercial recommendations by highlighting stages that warrant further investigation.

Particular attention is given to the checkout and payment stages, where differences in event-level user counts may indicate potential friction.

### Methodological Limitation

The SQL analysis counts distinct users independently for each event type. It does not verify that the same users completed all preceding stages in chronological order.

Therefore, differences between stages must not be interpreted as confirmed user abandonment or sequential conversion rates.

A future improvement would reconstruct individual user journeys using user identifiers and event timestamps to calculate validated stage-to-stage conversion and drop-off rates.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Database access is optional.
# Set environment variables only if authorised access is available.

required_vars = [
    "DB_USER",
    "DB_PASSWORD",
    "DB_HOST",
    "DB_PORT",
    "DB_NAME",
]

engine = None

if all(os.getenv(var) for var in required_vars):
    database_url = URL.create(
        drivername="postgresql+psycopg2",
        username=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        database=os.getenv("DB_NAME"),
    )

    engine = create_engine(
        database_url,
        connect_args={"sslmode": "require"}
    )
    print("Database connection configured.")
else:
    print("Database credentials unavailable. SQL queries can be reviewed without execution.")

In [ ]:
# Explore tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''

if engine is not None:
    events = pd.read_sql(query_events, con=engine)
    display(events.head())
else:
    print("SQL execution skipped: database access is unavailable.")


In [ ]:
# PART 1: Funnel Totals
# ======================

query_totals = '''
SELECT 
    nombre_evento AS paso_funnel,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM 
    events
GROUP BY 
    nombre_evento
ORDER BY 
    usuarios_unicos DESC;
'''


if engine is not None:
    totals = pd.read_sql(query_totals, con=engine)
    display(totals)
else:
    print("Funnel totals skipped: database access is unavailable.")


In [ ]:

# PART 2: Event-Level Funnel Comparisons
# =====================================

query_conversion = '''
WITH funnel_base AS (
    SELECT
        nombre_evento AS funnel_stage,
        COUNT(DISTINCT id_usuario) AS unique_users,
        CASE
            WHEN nombre_evento = 'first_visit' THEN 1
            WHEN nombre_evento = 'select_item' THEN 2
            WHEN nombre_evento = 'add_to_cart' THEN 3
            WHEN nombre_evento = 'begin_checkout' THEN 4
            WHEN nombre_evento = 'add_payment_info' THEN 5
            WHEN nombre_evento = 'purchase' THEN 6
        END AS stage_order
    FROM events
    WHERE nombre_evento IN (
        'first_visit',
        'select_item',
        'add_to_cart',
        'begin_checkout',
        'add_payment_info',
        'purchase'
    )
    GROUP BY nombre_evento
),
funnel_comparison AS (
    SELECT
        funnel_stage,
        unique_users,
        stage_order,
        LAG(unique_users) OVER (
            ORDER BY stage_order
        ) AS previous_stage_users,
        MAX(
            CASE WHEN stage_order = 1 THEN unique_users END
        ) OVER () AS initial_stage_users
    FROM funnel_base
)
SELECT
    funnel_stage,
    unique_users,
    stage_order,
    previous_stage_users,
    unique_users - previous_stage_users
        AS difference_vs_previous_stage,
    ROUND(
        100.0 * unique_users /
        NULLIF(previous_stage_users, 0), 2
    ) AS ratio_vs_previous_stage_pct,
    ROUND(
        100.0 * unique_users /
        NULLIF(initial_stage_users, 0), 2
    ) AS ratio_vs_initial_stage_pct
FROM funnel_comparison
ORDER BY stage_order;
'''

if engine is not None:
    conversion = pd.read_sql(query_conversion, con=engine)
    display(conversion)
else:
    print(
        "Event-level funnel comparison skipped: "
        "database access is unavailable."
    )

---


## 04. Customer Retention & Cohort Analysis

### 4.1 Registration Cohorts & Weekly Retention

**Objective:** Evaluate customer retention by measuring whether users return and remain active during the weeks following registration.

### Data Sources

The analysis uses two PostgreSQL tables:

- `users` — User registration information.
- `user_activity` — Records of user activity on the platform.

### Analytical Methodology

**1. Cohort Definition**

Users are grouped into cohorts based on their month of registration.

**2. Weekly Activity Measurement**

For each cohort, the analysis identifies users who were active during the first three weeks following registration:

- `retenido_w1` — Active users in Week 1.
- `retenido_w2` — Active users in Week 2.
- `retenido_w3` — Active users in Week 3.

**3. Retention Rate Calculation**

Weekly retention is calculated as:

Retention Rate (%) = (Active Users in Week N / Initial Cohort Size) × 100

The resulting metrics are:

- `semana_1` — Week 1 retention rate.
- `semana_2` — Week 2 retention rate.
- `semana_3` — Week 3 retention rate.

**4. Date Standardisation**

Registration dates are converted to the SQL `DATE` type using `CAST(fecha_registro AS DATE)` to support consistent cohort assignment.

### Business Application

Cohort retention analysis helps evaluate customer engagement after registration, identify changes in repeat activity, and inform customer retention strategies.


In [ ]:

# Explore tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''

if engine is not None:
    users = pd.read_sql(query_users, con=engine)
    display(users.head())
else:
    print("User data query skipped: database access is unavailable.")



In [ ]:
# Explore tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''

if engine is not None:
    user_activity = pd.read_sql(query_user_activity, con=engine)
    display(user_activity.head())
else:
    print("User activity query skipped: database access is unavailable.")


In [ ]:
# Cohort Retention
# ======================

query_cohort_retention_final = '''
WITH u_cohorts AS (
    -- Paso 1: CAST la fecha de registro
    SELECT 
        id_usuario,
        TO_CHAR(CAST(fecha_registro AS DATE), 'YYYY-MM') AS cohorte_mes
    FROM 
        users
),
user_weeks AS (
    -- Paso 2: Determinar en qué semana cae más la conversión
    SELECT 
        a.id_usuario,
        u.cohorte_mes,
        FLOOR(a.dias_despues_registro / 7) AS semana_actividad
    FROM 
        user_activity a
    JOIN 
        u_cohorts u ON a.id_usuario = u.id_usuario
    WHERE 
        a.activo = 1 -- Solo usuarios activos
),
cohort_sizes AS (
    -- Paso 3: conteo total de usuarios únicos que se unieron en el mes 
    SELECT 
        cohorte_mes,
        COUNT(DISTINCT id_usuario) AS usuarios_iniciales
    FROM 
        u_cohorts
    GROUP BY 
        cohorte_mes
),
retention_counts AS (
    -- Paso 4: Conteo de usuarios únicos activos que pasaron cada milestone (W1, W2, W3)
    SELECT 
        cohorte_mes,
        semana_actividad,
        COUNT(DISTINCT id_usuario) AS usuarios_activos
    FROM 
        user_weeks
    WHERE 
        semana_actividad BETWEEN 1 AND 3
    GROUP BY 
        cohorte_mes, 
        semana_actividad
)
-- Paso 5: Retencion en porcentaje
SELECT 
    c.cohorte_mes AS cohorte,
    c.usuarios_iniciales,
    
    -- Conteo Absoluto de usuarios retenidos
    COALESCE(MAX(CASE WHEN r.semana_actividad = 1 THEN r.usuarios_activos END), 0) AS retenido_w1,
    COALESCE(MAX(CASE WHEN r.semana_actividad = 2 THEN r.usuarios_activos END), 0) AS retenido_w2,
    COALESCE(MAX(CASE WHEN r.semana_actividad = 3 THEN r.usuarios_activos END), 0) AS retenido_w3,
    
    -- Porcentajes de retencion 
    TO_CHAR((COALESCE(MAX(CASE WHEN r.semana_actividad = 1 THEN r.usuarios_activos END), 0)::FLOAT / c.usuarios_iniciales) * 100, '999.99%%') AS semana_1,
    TO_CHAR((COALESCE(MAX(CASE WHEN r.semana_actividad = 2 THEN r.usuarios_activos END), 0)::FLOAT / c.usuarios_iniciales) * 100, '999.99%%') AS semana_2,
    TO_CHAR((COALESCE(MAX(CASE WHEN r.semana_actividad = 3 THEN r.usuarios_activos END), 0)::FLOAT / c.usuarios_iniciales) * 100, '999.99%%') AS semana_3
FROM 
    cohort_sizes c
LEFT JOIN 
    retention_counts r ON c.cohorte_mes = r.cohorte_mes
GROUP BY 
    c.cohorte_mes, 
    c.usuarios_iniciales
ORDER BY 
    c.cohorte_mes;
'''

# Execute cohort retention analysis when database access is available
if engine is not None:
    cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
    display(cohorte_final)
else:
    print("Cohort retention analysis skipped: database access is unavailable.")


---


## 05. A/B Testing & Statistical Experimentation

### 5.1 Checkout Interface Experiment

**Objective:** Evaluate whether changes to the checkout user interface are associated with a statistically significant difference in purchase conversion rates.

### Dataset & Primary Metric

The analysis uses `experiment_checkout_ui.csv`, containing results from a checkout interface experiment.

The primary metric is `convirtio`, a binary variable:

- `1` — User completed a purchase.
- `0` — User did not complete a purchase.

### Statistical Hypotheses

**Null Hypothesis (H₀):**

The purchase conversion rates are equal between the control and treatment groups.

H₀: p_control = p_treatment

**Alternative Hypothesis (H₁):**

The purchase conversion rates differ between the control and treatment groups.

H₁: p_control ≠ p_treatment

### Statistical Methodology

A **two-proportion Z-test** is used to compare the conversion rates of two independent groups.

**Significance level:** α = 0.05 (5%)

The analysis follows four stages:

1. Examine the experiment dataset and calculate conversion rates.
2. Define the null and alternative hypotheses.
3. Apply the two-proportion Z-test.
4. Interpret statistical significance and its implications for the checkout interface.

### Business Interpretation

The experiment evaluates whether the proposed interface change provides sufficient statistical evidence to support deployment.

The observed result (p ≈ 0.416) does not meet the 5% significance threshold. Therefore, the analysis does not find sufficient evidence of a difference in purchase conversion rates between the two variants.

This does not prove that both interfaces perform identically; it indicates that the experiment did not detect a statistically significant difference.


---

In [ ]:

import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

# 1. Load the A/B testing dataset from the local CSV
experiment_data = pd.read_csv("experiment_checkout_ui.csv")

# 2. Calculate conversions and total users by experiment group
summary = experiment_data.groupby("variante")["convirtio"].agg(["sum", "count"])
successes = summary["sum"].values
experiment_totals = summary["count"].values

print("--- A/B TEST: EXPERIMENT GROUP SUMMARY ---")
print(summary)
print("-" * 45)

# 3. Perform a two-sided, two-proportion Z-test
z_stat, p_value = proportions_ztest(
    count=successes,
    nobs=experiment_totals,
    alternative="two-sided"
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# 4. Interpret the results at a 5% significance level
alpha = 0.05

print("\n--- STATISTICAL CONCLUSION ---")

if p_value < alpha:
    print(
        f"P-value ({p_value:.4f}) < alpha ({alpha}). "
        "Reject the null hypothesis."
    )
    print(
        "Insight: There is a statistically significant difference "
        "in conversion rates between the two groups."
    )
else:
    print(
        f"P-value ({p_value:.4f}) >= alpha ({alpha}). "
        "Fail to reject the null hypothesis."
    )
    print(
        "Insight: The observed difference in conversion rates "
        "is not statistically significant."
    )

### A/B Testing — Statistical Interpretation

A two-sided, two-proportion Z-test was conducted to compare checkout conversion rates between the control and treatment groups.

- **Control:** 779 conversions out of 4,965 users (15.69%).
- **Treatment:** 820 conversions out of 5,035 users (16.29%).
- **Z-statistic:** −0.8133.
- **P-value:** 0.4161.
- **Significance level:** α = 0.05.

**Finding:** Although the treatment group achieved a slightly higher observed conversion rate, the difference was not statistically significant (p > 0.05).

**Business recommendation:** Do not roll out the checkout UI change based on these results alone. Further experimentation may be warranted before making a deployment decision.

---


## 06. Business Intelligence & Power BI Reporting

### 6.1 Dashboard Development & Data Modelling

**Objective:** Translate the analytical findings into interactive Power BI dashboards that support commercial performance monitoring and data-driven decision-making.

### Data Preparation

The reporting model uses three cleaned datasets:

- `orders_clean.csv` — Validated transactional data.
- `catalog_clean.csv` — Product catalogue and cost information.
- `marketing_clean.csv` — Marketing expenditure data.

The Power BI model connects transactional and product information through `nombre_producto`, with date relationships supporting time-based performance analysis.

### 6.2 Commercial Overview Dashboard

The executive overview presents:

- Total Revenue
- Total Profit
- Marketing Expenditure
- Average Order Value
- Average Products per Order
- Monthly Revenue and Year-to-Date Performance
- Revenue and Profit by Product Category

**Purpose:** Provide a consolidated view of financial performance, profitability, and commercial trends.

### 6.3 Commercial Details Dashboard

The detailed dashboard enables product-level performance analysis through:

- Product sales volumes.
- Revenue, costs, and profit by product.
- Conditional formatting to highlight profitability differences.
- Interactive filtering by country, reporting period, and product category.

**Purpose:** Identify commercial performance drivers and support product-level decision-making.

### 6.4 Commercial Insights Dashboard

The insights dashboard brings together customer behaviour, statistical experimentation, and business recommendations.

It includes:

- Customer conversion funnel.
- Identification of checkout and payment-stage friction.
- Checkout A/B test results.
- Statistical significance assessment.
- Commercial action recommendations.

**Experiment result:** p = 0.4161, indicating that the checkout interface experiment did not detect a statistically significant conversion difference at the 5% significance level.

**Purpose:** Connect analytical evidence with recommendations for customer experience and commercial optimisation.

### Business Impact

The three dashboards demonstrate how data preparation, SQL analysis, statistical testing, and business intelligence can be combined to evaluate commercial performance and identify opportunities for operational improvement.


---